# Transformer Best Translator

This notebook loads the newest saved `*_best.keras` checkpoint from `Transformers Outputs`, translates English into Igbo with that model, and gives you a quick Google Translate back-translation link to sanity-check the output.

Important: back-translation is only a rough comparison aid. Your saved transformer metrics and preview CSVs are still the more trustworthy record of model quality.

In [1]:
from __future__ import annotations

import importlib.util
import json
import re
import sys
from html import escape
from pathlib import Path
from urllib.parse import quote

if importlib.util.find_spec("tensorflow") is None:
    print("TensorFlow is not installed in this notebook kernel.")
    print()
    print("Run this in a new cell, then restart the kernel:")
    print("%pip install --upgrade pip")
    print("%pip install tensorflow ipywidgets")
    print()
    print("Native Windows TensorFlow is CPU-only for current releases.")
    print("If you want GPU execution, use Colab or WSL2 instead.")
    raise SystemExit("Install TensorFlow, restart the kernel, then rerun the notebook from the top.")

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from IPython.display import HTML, Markdown, display

if importlib.util.find_spec("ipywidgets") is not None:
    import ipywidgets as widgets
else:
    widgets = None
    print("ipywidgets is not installed. The manual translation cell will still work.")

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("TensorFlow version:", tf.__version__)
print("Detected GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
Detected GPUs: []


## 1. Find The Saved Best Checkpoint

The notebook looks in either `Transformers Outputs` or `Transformer Outputs` and picks the newest `*_best.keras` file.

In [2]:
PROJECT_DIR_CANDIDATES = [Path.cwd(), Path.cwd().parent]
OUTPUT_DIR_NAMES = ["Transformers Outputs", "Transformer Outputs"]


def resolve_project_dir(candidates):
    for candidate in candidates:
        for output_dir_name in OUTPUT_DIR_NAMES:
            if (candidate / output_dir_name).exists():
                return candidate
    return None


def resolve_output_dir(project_dir):
    for output_dir_name in OUTPUT_DIR_NAMES:
        output_dir = project_dir / output_dir_name
        if output_dir.exists():
            return output_dir
    raise FileNotFoundError("Could not locate a transformer output folder.")


PROJECT_DIR = resolve_project_dir(PROJECT_DIR_CANDIDATES)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project directory. Run this notebook from the project root or one level below it."
    )

OUTPUT_DIR = resolve_output_dir(PROJECT_DIR)
best_model_candidates = sorted(
    OUTPUT_DIR.glob("*_best.keras"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not best_model_candidates:
    raise FileNotFoundError(f"No *_best.keras checkpoint was found in {OUTPUT_DIR}.")

BEST_MODEL_PATH = best_model_candidates[0]
RUN_NAME = BEST_MODEL_PATH.name[: -len("_best.keras")]
CONFIG_PATH = OUTPUT_DIR / f"{RUN_NAME}_config.json"
SOURCE_VOCAB_PATH = OUTPUT_DIR / f"{RUN_NAME}_source_vocab.txt"
TARGET_VOCAB_PATH = OUTPUT_DIR / f"{RUN_NAME}_target_vocab.txt"

for required_path in [BEST_MODEL_PATH, CONFIG_PATH, SOURCE_VOCAB_PATH, TARGET_VOCAB_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required file is missing: {required_path}")

print("Project directory:", PROJECT_DIR)
print("Output directory:", OUTPUT_DIR)
print("Selected run name:", RUN_NAME)
print("Best model path:", BEST_MODEL_PATH)
print("Config path:", CONFIG_PATH)

Project directory: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment
Output directory: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\Transformers Outputs
Selected run name: transformer_all_baseline
Best model path: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\Transformers Outputs\transformer_all_baseline_best.keras
Config path: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\Transformers Outputs\transformer_all_baseline_config.json


## 2. Load The Saved Run Config And Vocabularies

The translator notebook rebuilds the token lookups from the saved vocab text files so inference matches the training run.

In [3]:
run_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

MAX_SOURCE_TOKENS = int(run_config["max_source_tokens"])
MAX_TARGET_TOKENS = int(run_config["max_target_tokens"])
TARGET_SEQUENCE_LENGTH = MAX_TARGET_TOKENS + 2
DECODE_BATCH_SIZE = 8
SUPPRESS_UNK_IN_DECODING = True

START_TOKEN = "[start]"
END_TOKEN = "[end]"

source_vocab = SOURCE_VOCAB_PATH.read_text(encoding="utf-8").splitlines()
target_vocab = TARGET_VOCAB_PATH.read_text(encoding="utf-8").splitlines()

source_token_lookup = {token: index for index, token in enumerate(source_vocab)}
target_token_lookup = {token: index for index, token in enumerate(target_vocab)}
target_index_lookup = dict(enumerate(target_vocab))

SOURCE_UNK_ID = source_token_lookup.get("[UNK]", 1)
TARGET_UNK_ID = target_token_lookup.get("[UNK]", None)

if START_TOKEN not in target_token_lookup or END_TOKEN not in target_token_lookup:
    raise ValueError("Saved target vocabulary is missing [start] or [end].")

START_TOKEN_ID = target_token_lookup[START_TOKEN]
END_TOKEN_ID = target_token_lookup[END_TOKEN]

print("Source vocab size:", len(source_vocab))
print("Target vocab size:", len(target_vocab))
print("Start token id:", START_TOKEN_ID)
print("End token id:", END_TOKEN_ID)
print("Suppress [UNK] during decoding:", SUPPRESS_UNK_IN_DECODING)

Source vocab size: 30000
Target vocab size: 30000
Start token id: 3
End token id: 4
Suppress [UNK] during decoding: True


## 3. Custom Objects Needed To Reload The Saved `.keras` Model

These definitions match the transformer training notebook so Keras can reconstruct the model exactly.

In [4]:
@keras.utils.register_keras_serializable(package="cs156")
def masked_loss(y_true, y_pred):
    loss = keras.losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)
    mask = tf.cast(tf.not_equal(y_true, 0), loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / tf.maximum(tf.reduce_sum(mask), 1.0)


@keras.utils.register_keras_serializable(package="cs156")
def masked_accuracy(y_true, y_pred):
    y_pred = tf.argmax(y_pred, axis=-1, output_type=y_true.dtype)
    matches = tf.cast(tf.equal(y_true, y_pred), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    matches *= mask
    return tf.reduce_sum(matches) / tf.maximum(tf.reduce_sum(mask), 1.0)


@keras.utils.register_keras_serializable(package="cs156")
class PaddingMask(layers.Layer):
    def call(self, inputs):
        return tf.not_equal(inputs, 0)

    def get_config(self):
        return super().get_config()


@keras.utils.register_keras_serializable(package="cs156")
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim,
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length,
            output_dim=embed_dim,
        )

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config


@keras.utils.register_keras_serializable(package="cs156")
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, dropout, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.dropout = dropout
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout,
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.dropout_1 = layers.Dropout(dropout)
        self.dropout_2 = layers.Dropout(dropout)

    def call(self, inputs, training=False, padding_mask=None):
        attention_mask = None
        if padding_mask is not None:
            attention_mask = tf.cast(padding_mask[:, tf.newaxis, :], dtype="bool")

        attention_output = self.attention(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=attention_mask,
            training=training,
        )
        attention_output = self.dropout_1(attention_output, training=training)
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        proj_output = self.dropout_2(proj_output, training=training)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
                "dropout": self.dropout,
            }
        )
        return config


@keras.utils.register_keras_serializable(package="cs156")
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, dropout, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.dropout = dropout
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout,
        )
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout,
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.dropout_1 = layers.Dropout(dropout)
        self.dropout_2 = layers.Dropout(dropout)
        self.dropout_3 = layers.Dropout(dropout)

    def get_causal_attention_mask(self, inputs):
        batch_size = tf.shape(inputs)[0]
        sequence_length = tf.shape(inputs)[1]
        mask = tf.linalg.band_part(
            tf.ones((sequence_length, sequence_length), dtype="bool"),
            -1,
            0,
        )
        mask = tf.expand_dims(mask, axis=0)
        return tf.tile(mask, [batch_size, 1, 1])

    def call(self, inputs, encoder_outputs, training=False, decoder_padding_mask=None, encoder_padding_mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)
        self_attention_mask = causal_mask

        if decoder_padding_mask is not None:
            padding_mask = tf.cast(decoder_padding_mask[:, tf.newaxis, :], dtype="bool")
            self_attention_mask = tf.logical_and(causal_mask, padding_mask)

        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=self_attention_mask,
            training=training,
        )
        attention_output_1 = self.dropout_1(attention_output_1, training=training)
        out_1 = self.layernorm_1(inputs + attention_output_1)

        cross_attention_mask = None
        if encoder_padding_mask is not None:
            cross_attention_mask = tf.cast(encoder_padding_mask[:, tf.newaxis, :], dtype="bool")

        attention_output_2 = self.attention_2(
            query=out_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=cross_attention_mask,
            training=training,
        )
        attention_output_2 = self.dropout_2(attention_output_2, training=training)
        out_2 = self.layernorm_2(out_1 + attention_output_2)

        proj_output = self.dense_proj(out_2)
        proj_output = self.dropout_3(proj_output, training=training)
        return self.layernorm_3(out_2 + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
                "dropout": self.dropout,
            }
        )
        return config


CUSTOM_OBJECTS = {
    "masked_loss": masked_loss,
    "masked_accuracy": masked_accuracy,
    "PaddingMask": PaddingMask,
    "PositionalEmbedding": PositionalEmbedding,
    "TransformerEncoder": TransformerEncoder,
    "TransformerDecoder": TransformerDecoder,
}

## 4. Load The Best Saved Model

In [5]:
translator_model = keras.models.load_model(BEST_MODEL_PATH, custom_objects=CUSTOM_OBJECTS)

print("Loaded model:", BEST_MODEL_PATH)
print("Run name:", RUN_NAME)
print("Training stage:", run_config.get("training_stage"))
print("Epochs trained:", run_config.get("epochs"))
print("Batch size used during training:", run_config.get("batch_size"))
print("Max source tokens:", MAX_SOURCE_TOKENS)
print("Max target tokens:", MAX_TARGET_TOKENS)

C:\Users\Mr. Paul\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\layer.py:427: UserWarning: `build()` was called on layer 'encoder_positional_embedding', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
C:\Users\Mr. Paul\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\layer.py:427: UserWarning: `build()` was called on layer 'encoder_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
C:\Users\Mr. Paul\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\lay


Loaded model: c:\Users\Mr. Paul\Downloads\CS156 Final Assignment\Transformers Outputs\transformer_all_baseline_best.keras
Run name: transformer_all_baseline
Training stage: all
Epochs trained: 8
Batch size used during training: 64
Max source tokens: 40
Max target tokens: 40


## 5. Translation Helpers

The helper below does batched greedy decoding with the same saved vocabulary. It also gives you a Google Translate link for Igbo-to-English back-translation.

In [6]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)


def normalize_text(text):
    text = str(text).lower().strip()
    text = text.replace("\u2019", "'")
    text = text.replace("\u2018", "'")
    text = text.replace("\u201c", '"')
    text = text.replace("\u201d", '"')
    text = text.replace("\u02bc", "'")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize(text):
    return TOKEN_PATTERN.findall(normalize_text(text))


def encode_source_batch(texts):
    encoded = np.zeros((len(texts), MAX_SOURCE_TOKENS), dtype="int64")

    for row_index, text in enumerate(texts):
        tokens = tokenize(text)[:MAX_SOURCE_TOKENS]
        token_ids = [source_token_lookup.get(token, SOURCE_UNK_ID) for token in tokens]
        encoded[row_index, : len(token_ids)] = token_ids

    return tf.convert_to_tensor(encoded, dtype=tf.int64)


def greedy_decode_batch(model, texts, max_decoded_tokens=MAX_TARGET_TOKENS, batch_size=DECODE_BATCH_SIZE):
    normalized_texts = [" ".join(tokenize(text)[:MAX_SOURCE_TOKENS]) for text in texts]
    all_predictions = []

    for start in range(0, len(normalized_texts), batch_size):
        batch_texts = normalized_texts[start:start + batch_size]
        encoder_inputs = encode_source_batch(batch_texts)
        current_batch_size = len(batch_texts)

        decoder_inputs = np.zeros((current_batch_size, TARGET_SEQUENCE_LENGTH - 1), dtype="int64")
        decoder_inputs[:, 0] = START_TOKEN_ID

        generated_token_ids = [[] for _ in range(current_batch_size)]
        finished = np.zeros(current_batch_size, dtype=bool)

        for step in range(max_decoded_tokens):
            decoder_inputs_tensor = tf.convert_to_tensor(decoder_inputs, dtype=tf.int64)
            predictions = model(
                {
                    "encoder_inputs": encoder_inputs,
                    "decoder_inputs": decoder_inputs_tensor,
                },
                training=False,
            )

            step_logits = predictions[:, step, :]
            if SUPPRESS_UNK_IN_DECODING and TARGET_UNK_ID is not None:
                blocked_ids = tf.constant([0, START_TOKEN_ID, TARGET_UNK_ID], dtype=tf.int32)
                blocked_mask = tf.reduce_sum(
                    tf.one_hot(blocked_ids, depth=tf.shape(step_logits)[-1], dtype=step_logits.dtype),
                    axis=0,
                )
                step_logits = step_logits - blocked_mask[tf.newaxis, :] * tf.cast(1e9, step_logits.dtype)

            next_token_ids = tf.argmax(step_logits, axis=-1, output_type=tf.int64).numpy()

            for row_index, token_id in enumerate(next_token_ids):
                if finished[row_index]:
                    continue

                token_id = int(token_id)
                if token_id in {0, END_TOKEN_ID}:
                    finished[row_index] = True
                    continue

                generated_token_ids[row_index].append(token_id)

                if step + 1 < decoder_inputs.shape[1]:
                    decoder_inputs[row_index, step + 1] = token_id

            if finished.all():
                break

        for token_ids in generated_token_ids:
            tokens = [target_index_lookup.get(token_id, "") for token_id in token_ids]
            tokens = [token for token in tokens if token and token not in {START_TOKEN, END_TOKEN}]
            all_predictions.append(" ".join(tokens))

    return all_predictions


def translate_text(text):
    text = str(text).strip()
    if not text:
        raise ValueError("Enter an English sentence first.")
    return greedy_decode_batch(translator_model, [text], batch_size=1)[0]


def google_translate_back_link(igbo_text):
    return f"https://translate.google.com/?sl=ig&tl=en&text={quote(igbo_text)}&op=translate"


def build_translation_html(english_text, igbo_text):
    back_link = google_translate_back_link(igbo_text)
    safe_english = escape(english_text)
    safe_igbo = escape(igbo_text if igbo_text else "[empty output]")
    return f"""
    <div style='font-family: sans-serif; line-height: 1.5;'>
      <p><strong>English input</strong></p>
      <div style='padding: 10px; border: 1px solid #ddd; border-radius: 6px; margin-bottom: 12px;'>{safe_english}</div>
      <p><strong>Transformer Igbo output</strong></p>
      <div style='padding: 10px; border: 1px solid #ddd; border-radius: 6px; margin-bottom: 12px;'>{safe_igbo}</div>
      <a href='{back_link}' target='_blank'>Open Google Translate back-translation (Igbo -> English)</a>
    </div>
    """


def show_translation_result(english_text):
    english_text = str(english_text).strip()
    if not english_text:
        raise ValueError("Enter an English sentence first.")

    igbo_text = translate_text(english_text)
    back_link = google_translate_back_link(igbo_text)
    display(HTML(build_translation_html(english_text, igbo_text)))

    return {
        "english_input": english_text,
        "igbo_output": igbo_text,
        "back_translation_url": back_link,
    }

## 6. Manual Translator Cell

Edit the sentence below and rerun the cell whenever you want a new translation.

In [7]:
english_text = "How are you?"

if english_text == "type your English sentence here":
    print("Replace `english_text` with a real sentence, then rerun this cell.")
else:
    result = show_translation_result(english_text)
    display(result)

{'english_input': 'How are you?',
 'igbo_output': 'olee otú i si eme ya ?',
 'back_translation_url': 'https://translate.google.com/?sl=ig&tl=en&text=olee%20ot%C3%BA%20i%20si%20eme%20ya%20%3F&op=translate'}

## 7. Optional Widget Tool

If `ipywidgets` is available, this gives you a small translator UI right inside the notebook.

In [8]:
if widgets is None:
    print("ipywidgets is not installed in this environment. Use the manual translator cell above instead.")
else:
    heading = widgets.HTML("<h3 style='margin: 0 0 8px 0;'>Transformer Translator</h3>")
    input_label = widgets.HTML("<b>English input</b>")
    input_box = widgets.Textarea(
        value="",
        placeholder="Type English text here...",
        layout=widgets.Layout(width="100%", height="120px"),
    )
    translate_button = widgets.Button(
        description="Translate",
        button_style="info",
        icon="language",
        tooltip="Run the saved transformer model on the input text",
    )
    status_box = widgets.HTML("")
    result_box = widgets.HTML("<i>Enter text above and click Translate.</i>")

    def on_translate_click(_):
        english_text = input_box.value.strip()
        if not english_text:
            status_box.value = "<span style='color: #b00020;'>Enter an English sentence first.</span>"
            result_box.value = ""
            return

        translate_button.disabled = True
        status_box.value = "<span style='color: #555;'>Translating...</span>"

        try:
            igbo_text = translate_text(english_text)
            result_box.value = build_translation_html(english_text, igbo_text)
            status_box.value = ""
        except Exception as exc:
            status_box.value = f"<span style='color: #b00020;'>Translation failed: {escape(str(exc))}</span>"
            result_box.value = ""
        finally:
            translate_button.disabled = False

    translate_button.on_click(on_translate_click)
    display(widgets.VBox([heading, input_label, input_box, translate_button, status_box, result_box]))